<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/Fine_Tune_Gemma_with_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**:

- Apply LoRA to Gemma-1B
- Fine-tune the model for own task
- Evaluate what the fine-tuned model learned from pre-training
- Evaluate what the fine-tuned model learned from fine-tuning
- Describe how combining knowledge from both training processes allows it to perform in ways that could not be achieved with either type of training alone

In [7]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os

from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

os.environ["KERAS_BACKEND"] = "jax"  # Set the Keras backend to JAX.

import json # For loading training logs.
from urllib import request # For downloading training logs.
import keras # For training the model.
import keras_hub # For loading Gemma-1B.
import pandas as pd # For loading the dataset.
import jax.numpy as jnp # For working with matrices and vectors.
from textwrap import fill # For formatting long paragraphs.
# For loading the formatting function that you implemented in previous labs.
from ai_foundations import formatting

# Avoids memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
keras.utils.set_random_seed(812)  # For making the training reproducible.

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-0vd15w57
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-0vd15w57
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Fine-tune Gemma 3 with LoRA**

- Data preparation and load the model

- format_qa: function to format question and answer

- format_question: formatting only a question;  
    - This function can be used for formatting prompts when you evaluate the model.

-

Steps:

- defines the format_question,
- loads the Africa Galore QA dataset,
- and creates the data dictionary that can be used to fine-tune the Keras implementation of Gemma

In [8]:
def format_question(
        question: str,
        sot: str = "<start of the turn>",
        eot: str = "<end of the turn>"
)-> str:

    format_q = f"{sot}user\n{question}{eot}\n"
    #formatted_q = f"{sot}user\n{question}{eot}\n"
    return format_q



In [9]:
# Load the question-answer dataset.
africa_galore_qa = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_v2.json"
)

questions = []  # List of formatted questions.
answers = []  # List of formatted answers.

for idx, row in africa_galore_qa.iterrows():
    # Run the format_qa function from the previous lab to format the question
    # and the answer.
    question, answer = formatting.format_qa(row)
    questions.append(question)
    answers.append(answer)

# Show the first set of input and output.
print(questions[0])
print(fill(answers[0], replace_whitespace=False))

# Prepare the data dictionary for fine-tuning Gemma.
data = {
    "prompts": questions,
    "responses": answers
}

<start_of_turn>user
What is Kente Cloth?<end_of_turn>

<start_of_turn>model
Category: Textile
The vibrant colors and
intricate patterns of Kente cloth, a symbol of Ghanaian royalty and
prestige, tell stories of history, culture, and social status. Woven
on narrow looms by skilled artisans, each strip of Kente is a
testament to patience and artistry. The geometric designs, rich with
symbolism, represent proverbs, historical events, and important
figures. Worn during special occasions and ceremonies, Kente cloth
embodies the spirit of Ghana, its vibrant culture, and its rich
history. From the bright yellows and golds representing royalty to the
deep blues and greens symbolizing spirituality, Kente is a visual
language, a wearable expression of Ghanaian identity and
heritage.<end_of_turn>


**Load the model**



In [10]:
# Load the Gemma-1B Keras model.
model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

**Prompt the pre-trained Gemma model**

- Before FT, observe the performance of pre-training model

In [11]:
evaluation_prompts = [
    "What is Kente cloth?",
    "What is the tallest mountain in Africa?",
    "What is Mount Aconcagua?"
]

# Predict answers for three formatted questions through the model.
# Generate answers with a length of (up to) 200 tokens.
for prompt in evaluation_prompts:
    formatted_prompt = format_question(prompt)
    model_response = model.generate(formatted_prompt, max_length=200)
    print(fill(model_response, replace_whitespace=False))
    print('\n------\n')

<start of the turn>user
What is Kente cloth?<end of the turn>
<start
of the turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>

------

<start of the turn>user
What is the tallest mountain in Africa?<end of
the turn>
<start of the turn>user
What is the tallest mountain in
Africa?<end of the turn>
<start of the turn>user
What is the tallest
mountain in Africa?<end of the turn>
<start of the turn>user
What is
the tallest mountain in Africa?<end of the turn>
<start of the
turn>user
What is the tallest mou

Repetitive questions and no answers.

**Activate LoRA**

- LoRA is already implemented in the Keras implementation of Gemma and can be added with a call to the enable_lora method


- to enable LoRA with a rank of 4, you can call:
model.backbone.enable_lora(rank = 4)


In [12]:
model.backbone.enable_lora(rank=4)
model.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,000,538,240 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,000,538,240 (3.73 GB)

 Trainable params: 652,288 (2.49 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

**Setting hyperparameters**

- The last step before you can start training is to set the hyperparameters of the training process.

    - two parameters that are particularly important for fine-tuning are the
        - learning rate  
        - the number of epochs

- For both of these parameters, want to perform hyperparameter tuning and
- choose a set of parameters that leads to useful results on evaluation examples.

- For pre-trained models like Gemma,
    - a reasonable range for the number of epochs is usually 1 to 20, depending both on the size of your fine-tuning set and the learning rate

    - If use a larger fine-tuning set and/or a higher learning rate, generally need to fine-tune for fewer epochs.

    - if have very few fine-tuning examples or use a very low learning rate, will need to fine-tune for more epochs


- For the learning rate,
    - values between 0.0001 (1e-4) and 0.000001 (1e-6) tend to work well in practice for fine-tuning models like Gemma




**Training logs for different learning rates**

-

In [20]:
if "training_logs" not in globals():
    TRAINING_LOG_URL = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/finetune-gemma-training-logs.json"
    with request.urlopen(TRAINING_LOG_URL) as json_file:
        training_logs = json.loads(json_file.read().decode())

learning_rate = 1e-4 # @param ["1e-3","5e-4","2e-4","1e-4","5e-5","2e-5","1e-5","5e-6","2e-6","1e-6"] {"type":"raw"}

print(training_logs[str(learning_rate)])

Epoch:1

130/130 ━━━━━━━━━━━━━━━━━━━━ 117s 655ms/step - loss: 0.5503 - sparse_categorical_accuracy: 0.5903
<start_of_turn>user
What is Kente Cloth?<end_of_turn>
<start_of_turn>model
Category: Textile
Kente cloth is a symbol of Ghana's rich cultural heritage, woven by skilled artisans in the Ashanti Kingdom. It is renowned for its intricate patterns and vibrant colors, which are derived from the natural dyes extracted from plants and insects. The cloth is often worn as a ceremonial garment, symbolizing power, wealth, and social status. It is a testament to the artistry and craftsmanship of the region's artisans, and it continues to be a cherished symbol of tradition and beauty.<end_of_turn>


------


<start_of_turn>user
What is Kilimanjaro?<end_of_turn>
<start_of_turn>model
Category: Natural
Kilimanjaro is the highest freestanding mountain in the world, and the tallest mountain in Africa. It is a dormant volcano, and the highest point in the world is the summit, which is 5,895 meters (

In [21]:
# Set the learning rate
model.optimizer.learning_rate = 1e-4

# Set the number of epochs.
num_epochs = 10

# Set the maximum length.
model.preprocessor.sequence_length = 400

**Perform fine-tuning**

- fine-tuning can be done just like any other training in Keras, that is, with the model.fit() method

- important to monitor the training progress using a callback function.

**Define a callback function that prints generations for the three evaluation prompts defined above.**

In [22]:
class EvaluationCallback(keras.callbacks.Callback):
    """
    A Keras callback function to print generations for evaluation prompts.
    """
    def on_epoch_end(self, epoch: int, logs=None):
        """Prints generations for the three evaluation prompts.

        Args:
          epoch: The current epoch.
          logs: The logs dictionary.
        """

        evaluation_prompts = [
            "What is Kente cloth?",
            "What is the tallest mountain in Africa?",
            "What is Mount Aconcagua?"
        ]

        # Run three formatted questions through the model.
        # Generate answers with a length of (up to) 200 tokens.
        for prompt in evaluation_prompts:
            formatted_prompt = format_question(prompt)
            model_response = model.generate(formatted_prompt, max_length=200)
            print(fill(model_response, replace_whitespace=False))
            print('\n------\n')


evaluation_callback = EvaluationCallback() # create an object of the class.
                        # Keras will automatically call:evaluation_callback.on_epoch_end


**Fine-tuning and monitor the fine-tuning progress**

- Monitor:

    - When does the model begin to output "<start_of_turn>model"?

    - When does the model begin to produce outputs that start with "Category:" and the correct category?

    - At which epochs does the model produce very strange outputs?

    - From which epoch on does the model consistently produce a single paragraph and stop repeating texts?

    - At which epoch does the model start to produce "<end_of_turn>" and stop producing any other output thereafter?

In [23]:
training_history = model.fit(
    data,
    epochs=num_epochs,
    batch_size=1,
    verbose=1,
    callbacks=[evaluation_callback]
)

Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - loss: 0.6832 - sparse_categorical_accuracy: 0.5535<start of the turn>user
What is Kente cloth?<end of the turn>
<start
of the turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>
<start of the
turn>user
What is Kente cloth?<end of the turn>

------

<start of the turn>user
What is the tallest mountain in Africa?<end of
the turn>
<start of the turn>user
What is the tallest mountain in
Africa?<end of the turn>
<start of the turn>user
What is the tallest
mountain in Africa?<end of the turn>
<start of the turn>use

In [24]:
more_epochs = 5

training_history = model.fit(
    data,
    epochs=more_epochs,
    batch_size=1,
    verbose=1,
    callbacks=[evaluation_callback]
)

Epoch 1/5
129/130 ━━━━━━━━━━━━━━━━━━━━ 0s 521ms/step - loss: 0.3376 - sparse_categorical_accuracy: 0.6971<start of the turn>user
What is Kente cloth?<end of the turn>
<start
of the turn>user
Kente cloth is a traditional West African textile
from the Ashanti Kingdom in modern-day Ghana. It is made from a
tightly woven silk warp and a loosely woven weft of cotton, and is
characterized by its bold, geometric patterns and rich, deep colors.
The cloth is usually rectangular in shape and is worn as a garment,
often worn by royalty and other important figures. Kente cloth is
known for its durability and intricate patterns, which often depict
stories, animals, and symbols from Ashanti culture. It is a symbol of
prestige and status in West Africa, and is still widely produced and
worn today. <end of the turn>
<start of the turn>user
<start of the
turn>user
<start of the turn>user
<start of the turn>user
<start of
the turn>user
<start of

------

<start of the turn>user
What is the tallest mount

**Additional evaluations**

-

In [25]:
additional_prompts = [
    "What is Mount Aconcagua?",
    "What is American Football?",
    "What is Python?"
]

for prompt in additional_prompts:
    formatted_prompt = format_question(prompt)
    model_response = model.generate(formatted_prompt, max_length=300)
    print(fill(model_response, replace_whitespace=False))
    print('\n------\n')

<start of the turn>user
What is Mount Aconcagua?<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end
of the turn>
<start of the turn>user
<end of the turn>
<start of the
turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end
of the turn>
<start of the turn>user
<end of the turn>
<start of the
turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end
of the turn>
<start of the turn>user
<end of the turn>
<start of the
turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end of the turn>
<start of the turn>user
<end
of the turn>
<start of the turn>user
<end of the turn>
<start of the
turn>user

------

<start of the turn>user
What is American Football?<end of the turn>
<start of the turn>player
I'm a football player. I play for the
University of

**Observations**

- training loss keeps decreasing, but sparse categorical accuracy keeps increasing,and generated answers get worse

- This usually means the model is getting better at predicting the training tokens, but not necessarily better at answering questions naturally.
- the metrics are improving, but the model is becoming less useful.

- likely causes:
    - 1. Overfitting: fine-tuned for too long on a small datase
    - 2. Catastrophic forgetting:the model can “forget” some of its original general-language ability, especially if the learning rate is too large or you train for too many epochs.
    - 3. Training accuracy != generability

- Counter measures:
    - 1. save checkpoints
    - 2. reduce learning rate to 5e-5 or 5e-6, too agressive lr would demage model's original knowledge and generalbity
    - 3. fewer ephocs
    - 4. seperate dataset to test and validate, if test loss is decreasing but validate loss is increasing, then overfit.
